[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/myway0101/see-read-explain-dl/blob/main/notebooks/ch03_cnn_basics.ipynb)

# 3장 합성곱 신경망(CNN) — 실습 노트북
『보고, 읽고, 설명하는 딥러닝 — Colab에서 직접 만들며 배우는 딥러닝 응용』

3장 본문에 나오는 코드를 순서대로 실행해 봅니다. 위에서부터 차례로 실행하세요.
- 3.3 합성곱 연산 직접 계산하기
- 3.4 출력 크기 공식 확인하기
- 3.6 파라미터 수 손으로 계산하기
- 3.7 풀링
- 3.9 케라스로 CNN 만들기: MNIST, Fashion-MNIST
- 실습: Fashion-MNIST 92% 넘기기 · 파라미터 수 · 완전연결 신경망과 비교

> **GPU를 켜고 실행하세요.** 메뉴 [런타임] → [런타임 유형 변경]에서 하드웨어 가속기를 **T4 GPU**로 고르면 3.9절과 실습이 훨씬 빨리 끝납니다.

## 준비

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
%config InlineBackend.figure_format = 'retina'

tf.keras.utils.set_random_seed(42)
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## 3.3 합성곱 연산 직접 계산하기
6×6 입력에 3×3 필터를 보폭 1로 움직이며 곱하고 더해 4×4 특성 맵을 만듭니다.

In [ ]:
X = np.array([[3, 0, 1, 2, 7, 4],
              [1, 5, 8, 9, 3, 1],
              [2, 7, 2, 5, 1, 3],
              [0, 1, 3, 1, 7, 8],
              [4, 2, 1, 6, 2, 8],
              [2, 4, 5, 2, 3, 9]])
K = np.array([[1, 0, -1],
              [1, 0, -1],
              [1, 0, -1]])

def conv2d(x, k, stride=1):
    kh, kw = k.shape
    oh = (x.shape[0] - kh) // stride + 1
    ow = (x.shape[1] - kw) // stride + 1
    out = np.zeros((oh, ow), dtype=np.result_type(x, k))
    for i in range(oh):
        for j in range(ow):
            window = x[i*stride:i*stride+kh, j*stride:j*stride+kw]
            # 같은 자리끼리 곱해서 모두 더한다
            out[i, j] = np.sum(window * k)
    return out

print(conv2d(X, K))

케라스의 `Conv2D` 층에 같은 필터를 넣어도 결과가 같은지 확인합니다.

In [ ]:
layer = tf.keras.layers.Conv2D(1, (3, 3), use_bias=False)
x_in = X.reshape(1, 6, 6, 1).astype('float32')       # (배치, 높이, 너비, 채널)
layer.build(x_in.shape)
layer.set_weights([K.reshape(3, 3, 1, 1).astype('float32')])
print(layer(x_in).numpy()[0, :, :, 0])

### 필터가 찾는 것: 손글씨 숫자에 세로·가로 경계 필터 적용하기

In [ ]:
(mnist_x, _), _ = tf.keras.datasets.mnist.load_data()
digit = mnist_x[0].astype('float32') / 255.0

filters = {'vertical edge': np.array([[1, 0, -1]] * 3),
           'horizontal edge': np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]])}
plt.figure(figsize=(9, 3))
plt.subplot(1, 3, 1); plt.imshow(digit, cmap='gray'); plt.title('input'); plt.axis('off')
for i, (name, k) in enumerate(filters.items()):
    plt.subplot(1, 3, i + 2)
    out = conv2d(digit, k); v = abs(out).max()
    plt.imshow(out, cmap='bwr', vmin=-v, vmax=v)      # 0을 흰색으로
    plt.title(name); plt.axis('off')
plt.show()

## 3.4 출력 크기 공식 확인하기
$\text{출력} = \lfloor (N - F + 2P)/S \rfloor + 1$

In [ ]:
def out_size(n, f, p, s):
    return (n - f + 2 * p) // s + 1

x28 = tf.zeros((1, 28, 28, 1))
for padding, p in [('valid', 0), ('same', 1)]:
    for s in [1, 2]:
        y = tf.keras.layers.Conv2D(8, (3, 3), strides=s, padding=padding)(x28)
        print(f"padding={padding:5s} stride={s} → 케라스 {tuple(y.shape[1:3])},  공식 {out_size(28, 3, p, s)}")

`padding='same'`에 보폭 2를 주면 케라스는 출력 크기를 입력 크기 ÷ 보폭(올림)으로 맞춥니다(28 → 14).

## 3.5–3.6 채널과 필터, 파라미터 수
컬러 입력(채널 3)에 필터 2개를 쓰면 출력 채널이 2가 됩니다. 파라미터 수는 $F \times F \times C \times K + K$ 입니다.

In [ ]:
def conv_params(f, c, k):
    return f * f * c * k + k          # 가중치 + 편향

color = tf.keras.layers.Conv2D(2, (3, 3))
print("출력 모양:", color(tf.zeros((1, 4, 4, 3))).shape)
print("케라스 파라미터:", color.count_params(), " / 공식:", conv_params(3, 3, 2))

In [ ]:
fc = tf.keras.layers.Dense(32)
fc.build((None, 28 * 28))
conv = tf.keras.layers.Conv2D(32, (3, 3))
conv.build((None, 28, 28, 1))
print("28×28 흑백 이미지 입력")
print("  완전연결층(노드 32개):", fc.count_params())
print("  합성곱층(3×3 필터 32개):", conv.count_params())

## 3.7 풀링

In [ ]:
P = np.array([[ 3, -1, 12, -1],
              [-3,  1,  0,  1],
              [ 2, -3,  0,  1],
              [ 3, -2,  4, -1]], dtype='float32')
x_in = P.reshape(1, 4, 4, 1)
print("최대 풀링\n", tf.keras.layers.MaxPooling2D((2, 2))(x_in).numpy()[0, :, :, 0])
print("평균 풀링\n", tf.keras.layers.AveragePooling2D((2, 2))(x_in).numpy()[0, :, :, 0])

## 3.9 케라스로 CNN 만들기
### MNIST

In [ ]:
from tensorflow.keras import datasets, layers, models

(train_images, train_labels), (test_images, test_labels) = \
    datasets.mnist.load_data()
train_images = train_images.reshape((60000, 28, 28, 1))
test_images = test_images.reshape((10000, 28, 28, 1))
# 픽셀 값을 0~1 사이로 정규화한다.
train_images, test_images = train_images / 255.0, test_images / 255.0

In [ ]:
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu',
                        input_shape=(28, 28, 1)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10, activation='softmax'))
model.summary()

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.fit(train_images, train_labels, epochs=5)

In [ ]:
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)
print('테스트 정확도:', test_acc)

### 학습된 첫 번째 합성곱층이 만든 특성 맵 보기

In [ ]:
feature_model = models.Model(inputs=model.inputs, outputs=model.layers[0].output)
maps = feature_model.predict(test_images[:1], verbose=0)[0]     # (26, 26, 32)
plt.figure(figsize=(10, 2.6))
plt.subplot(2, 9, 1); plt.imshow(test_images[0, :, :, 0], cmap='gray'); plt.title('input', fontsize=8); plt.axis('off')
for i in range(16):
    plt.subplot(2, 9, i + 2 + (i >= 8))
    plt.imshow(maps[:, :, i], cmap='viridis'); plt.axis('off')
plt.show()

### Fashion-MNIST

In [ ]:
(x_train, y_train), (x_test, y_test) = \
    tf.keras.datasets.fashion_mnist.load_data()

X_train_final = x_train.reshape((-1, 28, 28, 1)) / 255.
X_test_final = x_test.reshape((-1, 28, 28, 1)) / 255.

In [ ]:
model_with_conv = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu',
                           input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D((2, 2), strides=2),
    tf.keras.layers.Conv2D(64, (3, 3), padding='same',
                           activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2), strides=2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model_with_conv.compile(optimizer='adam',
                        loss='sparse_categorical_crossentropy',
                        metrics=['accuracy'])
model_with_conv.summary()

In [ ]:
model_with_conv.fit(X_train_final, y_train, epochs=5)
model_with_conv.evaluate(X_test_final, y_test, verbose=2)

## 실습: Fashion-MNIST 테스트 정확도 92% 넘기기
1. 아래 구조의 CNN을 만들어 테스트 정확도 92% 이상을 목표로 학습하고, 과적합이 생기지 않았는지 학습 곡선으로 확인합니다.
   - (3×3 합성곱층 2개 + 최대 풀링층 + 드롭아웃 0.3) × 3번, 필터 수는 32 → 64 → 128
   - Flatten → Dense(512) → Dropout(0.3) → 출력층
   - 학습 데이터의 20%를 검증 세트로, 검증 손실 기준 조기 종료(patience 10), 최고 모델 저장
   - 최대 200에포크, 배치 크기 256
2. 이 모델의 학습 파라미터 수를 층별로 직접 계산해 `summary()`와 비교합니다.
3. 파라미터 수가 비슷한 완전연결 신경망(은닉층 3개 × 512노드, 각 은닉층 뒤 드롭아웃 0.3)을 같은 조건으로 학습해 정확도와 파라미터 수를 비교합니다.

먼저 직접 작성해 본 뒤, 아래 예시 답안과 비교해 보세요.

In [ ]:
# 직접 작성해 보세요



### 예시 답안 1: CNN

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

def conv_block(filters):
    return [layers.Conv2D(filters, (3, 3), padding='same', activation='relu'),
            layers.Conv2D(filters, (3, 3), padding='same', activation='relu'),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.3)]

cnn = models.Sequential([layers.Input(shape=(28, 28, 1))]
                        + conv_block(32) + conv_block(64) + conv_block(128)
                        + [layers.Flatten(),
                           layers.Dense(512, activation='relu'),
                           layers.Dropout(0.3),
                           layers.Dense(10, activation='softmax')])
cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.summary()

In [ ]:
callbacks = [EarlyStopping(monitor='val_loss', patience=10),
             ModelCheckpoint('best_cnn.keras', monitor='val_loss', save_best_only=True)]
hist_cnn = cnn.fit(X_train_final, y_train, validation_split=0.2, epochs=200, batch_size=256,
                   callbacks=callbacks, verbose=2)

In [ ]:
best_cnn = models.load_model('best_cnn.keras')
cnn_acc = best_cnn.evaluate(X_test_final, y_test, verbose=0)[1]
print('CNN 테스트 정확도:', cnn_acc, ' / 학습한 에포크:', len(hist_cnn.history['loss']))

def plot_history(h, title):
    ep = range(1, len(h['loss']) + 1)
    plt.figure(figsize=(10, 3.5))
    plt.subplot(1, 2, 1); plt.plot(ep, h['loss'], label='train'); plt.plot(ep, h['val_loss'], label='validation')
    plt.title(title + ' loss'); plt.xlabel('epoch'); plt.legend()
    plt.subplot(1, 2, 2); plt.plot(ep, h['accuracy'], label='train'); plt.plot(ep, h['val_accuracy'], label='validation')
    plt.title(title + ' accuracy'); plt.xlabel('epoch'); plt.legend()
    plt.show()

plot_history(hist_cnn.history, 'CNN')

### 예시 답안 2: 파라미터 수 직접 계산

In [ ]:
shapes = [(1, 32), (32, 32), (32, 64), (64, 64), (64, 128), (128, 128)]
total = 0
for c, k in shapes:
    p = conv_params(3, c, k); total += p
    print(f"Conv2D 3×3, 입력 채널 {c:3d} → 필터 {k:3d}: {p:7,d}")
flat = 3 * 3 * 128                       # 28 → 14 → 7 → 3 (풀링 3번)
dense = flat * 512 + 512; out = 512 * 10 + 10
print(f"Dense {flat} → 512: {dense:,d}\nDense 512 → 10: {out:,d}")
total += dense + out
print(f"합계 {total:,d}  /  케라스 count_params(): {cnn.count_params():,d}")

### 예시 답안 3: 파라미터 수가 비슷한 완전연결 신경망

In [ ]:
fnn = models.Sequential([layers.Input(shape=(28, 28, 1)), layers.Flatten()]
                        + sum([[layers.Dense(512, activation='relu'), layers.Dropout(0.3)] for _ in range(3)], [])
                        + [layers.Dense(10, activation='softmax')])
fnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
callbacks = [EarlyStopping(monitor='val_loss', patience=10),
             ModelCheckpoint('best_fnn.keras', monitor='val_loss', save_best_only=True)]
hist_fnn = fnn.fit(X_train_final, y_train, validation_split=0.2, epochs=200, batch_size=256,
                   callbacks=callbacks, verbose=2)
fnn_acc = models.load_model('best_fnn.keras').evaluate(X_test_final, y_test, verbose=0)[1]
plot_history(hist_fnn.history, 'FNN')
print(f"CNN: 파라미터 {cnn.count_params():,d}개, 테스트 정확도 {cnn_acc:.4f}")
print(f"FNN: 파라미터 {fnn.count_params():,d}개, 테스트 정확도 {fnn_acc:.4f}")